In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import umap
import hdbscan
import matplotlib.pyplot as plt
import pickle
from tqdm.notebook import tqdm

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from scipy.stats import spearmanr

In [ ]:
cur_dir = os.getcwd().split('\\')

if cur_dir[-1] == 'notebooks':
    os.chdir("..")

from utils.data_load_utilities.data_loader import load_model_results
from utils.get_global_const import get_global_const
from utils.get_metrics import get_metrics
from utils.get_ranks import get_ranks_s, get_ranks
from methods.ADoE_method import *
from methods.k_nearest_methods import *
from methods.kmeans_methods import *
from methods.opt_methods import *
from methods.sparce_methods import *
from methods.entrophy_metods import *
from testing_pipeline.testing_pipeline import *

from sklearn.linear_model import LassoCV
from sklearn.feature_selection import mutual_info_regression

from datetime import datetime
import re

import warnings
warnings.simplefilter('ignore')

In [ ]:
def series_to_dataset_mean(in_csv, out_csv):
    
    df = pd.read_csv(in_csv)

    if "dataset" not in df.columns or "series_id" not in df.columns:
        raise ValueError("Expected columns: dataset, series_id")

    ds = df.drop(columns=["series_id"]).groupby("dataset", as_index=False).mean(numeric_only=True)

    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    ds.to_csv(out_csv, index=False)

    return ds

In [ ]:
def series_to_dataset_mean_chunked(
    in_csv,
    out_csv,
    chunksize=50_000,
):
    sums = None
    counts = None

    for chunk in tqdm(pd.read_csv(in_csv, chunksize=chunksize)):
        
        chunk = chunk.drop(columns=["series_id"])
        
        g = chunk.groupby("dataset")

        s = g.sum(numeric_only=True)
        c = g.count()

        if sums is None:
            sums = s
            counts = c
        else:
            sums = sums.add(s, fill_value=0)
            counts = counts.add(c, fill_value=0)

    ds = (sums / counts).reset_index()
    
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    
    ds.to_csv(out_csv, index=False)
    
    return ds

IN_DIR = Path("data/datasets_features")
OUT_DIR = IN_DIR / "dataset_level"

# series_to_dataset_mean(IN_DIR / "tsfresh_series_features.csv",   OUT_DIR / "tsfresh_dataset_mean.csv")
# series_to_dataset_mean(IN_DIR / "catch22_series_features.csv",   OUT_DIR / "catch22_dataset_mean.csv")
# series_to_dataset_mean_chunked(IN_DIR / "minirocket_series_features.csv", OUT_DIR / "minirocket_dataset_mean.csv")
# series_to_dataset_mean(IN_DIR / "summary_series_features.csv",   OUT_DIR / "summary_dataset_mean.csv")


In [ ]:
def _upper_tri_vals(M):
    iu = np.triu_indices_from(M, k=1)
    return M[iu]

In [ ]:
def compute_rank_distance_matrix(ranks_aggr):

    assert "dataset" in ranks_aggr.columns, "ranks_aggr must contain 'dataset' column"

    r = ranks_aggr.copy()
    
    r["dataset"] = r["dataset"].astype(str)

    if r["dataset"].duplicated().any():
        dup = r.loc[r["dataset"].duplicated(), "dataset"].tolist()[:10]
        raise ValueError(f"ranks_aggr has duplicated dataset names, examples: {dup}")

    datasets_order = r["dataset"].to_numpy()

    model_cols = [c for c in r.columns if c != "dataset"]
    
    R = r[model_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    
    R = np.nan_to_num(R, nan=0.0, posinf=0.0, neginf=0.0)

    n = R.shape[0]
    
    Corr = np.eye(n, dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            
            rho = spearmanr(R[i], R[j]).correlation
            
            if not np.isfinite(rho):
                rho = 0.0
                
            Corr[i, j] = Corr[j, i] = float(rho)

    D_rank = 1.0 - Corr
    return D_rank, datasets_order

In [ ]:
def compute_feature_distance_matrix(
    feats_ds,
    datasets_order,
    metric="cosine",
    scale=True,
    ):
    
    assert "dataset" in feats_ds.columns, "feats_ds must contain 'dataset' column"
    
    f = feats_ds.copy()
    
    f["dataset"] = f["dataset"].astype(str)

    if f["dataset"].duplicated().any():
        dup = f.loc[f["dataset"].duplicated(), "dataset"].tolist()[:10]
        raise ValueError(f"Feature table has duplicated dataset names, examples: {dup}")

    f = f.set_index("dataset").reindex(datasets_order)

    missing = f.index[f.isnull().all(axis=1)].tolist()
    
    if len(missing) > 0:
        raise ValueError(f"Some datasets from ranks_aggr are missing in features: {missing[:10]}")

    X = f.to_numpy(dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if scale:
        X = StandardScaler().fit_transform(X)

    return pairwise_distances(X, metric=metric)

In [ ]:
def feature_rank_alignment(
    feats_ds,
    ranks_aggr,
    feat_metric="cosine",
    feat_scale=True,
    ):
    
    D_rank, datasets_order = compute_rank_distance_matrix(ranks_aggr)
    
    D_feat = compute_feature_distance_matrix(feats_ds, datasets_order, metric=feat_metric, scale=feat_scale)

    v_rank = _upper_tri_vals(D_rank)
    
    v_feat = _upper_tri_vals(D_feat)

    corr = spearmanr(v_rank, v_feat).correlation
    
    if not np.isfinite(corr):
        corr = 0.0

    return {
        "spearman_corr(distance_rank, distance_feat)": float(corr),
        "n_datasets": int(len(datasets_order)),
        "feat_metric": feat_metric,
        "feat_scaled": bool(feat_scale),
        "p_value": spearmanr(v_rank, v_feat).pvalue
    }

In [7]:
scores, datasets, models = get_global_const()
scores

Parsing Bakeoff2017 models...

Parsing Bakeoff2021 models...

Parsing Bakeoff2023 models...

Parsing HIVE-COTEV2 models...



{'1NN-DTW':                   folds:         0         1         2         3         4  \
 0                  Adiac  0.603581  0.603581  0.601023  0.606138  0.608696   
 1              ArrowHead  0.702857  0.742857  0.680000  0.708571  0.760000   
 2                   Beef  0.633333  0.533333  0.533333  0.500000  0.566667   
 3              BeetleFly  0.700000  0.800000  0.650000  0.800000  0.650000   
 4            BirdChicken  0.750000  0.850000  0.950000  0.800000  0.950000   
 ..                   ...       ...       ...       ...       ...       ...   
 107  EOGHorizontalSignal  0.441989  0.607735  0.566298  0.569061  0.624309   
 108    EOGVerticalSignal  0.433702  0.511050  0.549724  0.566298  0.563536   
 109                 Rock  0.540000  0.680000  0.660000  0.640000  0.720000   
 110                 Crop  0.710536  0.713631  0.706964  0.708929  0.712321   
 111            Chinatown  0.973761  0.909621  0.953353  0.944606  0.970845   
 
             5         6         7     

In [8]:
tsfresh_features = pd.read_csv(Path('data/datasets_features/tsfresh_important_features.csv'), index_col=0)
chosen_datasets = datasets

In [9]:
ranks = get_ranks_s(chosen_datasets, scores, datasets, return_ranks=True)

In [10]:
ranks_aggr = pd.DataFrame(columns=chosen_datasets)

for d in chosen_datasets:
    ranks_aggr[d] = ranks[d].drop(columns=['model']).mean(axis=1)

ranks_aggr = ranks_aggr.transpose()

In [11]:
IN_DIR = Path("data/datasets_features")
OUT_DIR = Path("data/datasets_features/dataset_level")

In [12]:
ranks_aggr['dataset'] = ranks_aggr.index
ranks_aggr

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,dataset
Adiac,2.000000,16.066667,11.700000,15.733333,1.000000,6.666667,29.066667,3.933333,27.966667,22.133333,...,3.600000,6.633333,13.466667,18.333333,8.066667,20.933333,31.133333,22.833333,11.733333,Adiac
ArrowHead,2.666667,21.166667,20.200000,11.800000,5.600000,4.800000,13.066667,19.200000,7.900000,25.433333,...,18.433333,5.800000,29.900000,26.300000,7.000000,2.433333,27.333333,24.433333,25.733333,ArrowHead
Beef,3.200000,22.766667,11.600000,27.466667,15.766667,3.166667,28.066667,4.466667,28.166667,20.133333,...,4.766667,11.933333,14.166667,10.633333,14.900000,20.700000,25.400000,19.800000,7.966667,Beef
BeetleFly,5.166667,17.500000,24.566667,14.900000,8.566667,14.066667,18.500000,6.233333,18.366667,28.766667,...,4.300000,6.533333,23.333333,27.000000,9.900000,1.000000,25.533333,22.366667,29.600000,BeetleFly
BirdChicken,9.633333,16.633333,27.800000,12.066667,4.300000,14.700000,25.400000,12.300000,21.866667,27.633333,...,3.366667,3.800000,24.966667,23.366667,8.733333,4.866667,18.466667,17.800000,33.166667,BirdChicken
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
EOGHorizontalSignal,2.033333,24.166667,6.500000,20.200000,1.000000,11.566667,23.566667,20.800000,17.666667,23.100000,...,3.733333,8.333333,10.900000,31.666667,7.933333,14.800000,14.500000,7.133333,9.866667,EOGHorizontalSignal
EOGVerticalSignal,2.766667,26.233333,7.433333,24.266667,1.000000,11.800000,27.600000,19.600000,14.066667,23.566667,...,2.900000,6.466667,13.666667,32.533333,10.533333,12.166667,13.533333,3.533333,8.333333,EOGVerticalSignal
Rock,4.733333,16.533333,16.400000,20.266667,9.633333,12.266667,23.733333,12.200000,22.900000,25.266667,...,6.733333,10.533333,26.200000,20.500000,12.066667,6.900000,27.300000,23.000000,18.333333,Rock
Crop,6.700000,13.633333,2.100000,32.233333,5.833333,2.266667,33.466667,11.400000,27.766667,27.666667,...,7.866667,17.100000,8.966667,23.300000,15.766667,19.100000,21.033333,12.766667,4.266667,Crop


In [13]:
tsf = pd.read_csv("data/datasets_features/dataset_level/tsfresh_dataset_mean.csv")
c22 = pd.read_csv("data/datasets_features/dataset_level/catch22_dataset_mean.csv")
mr  = pd.read_csv("data/datasets_features/dataset_level/minirocket_dataset_mean.csv")
sm  = pd.read_csv("data/datasets_features/dataset_level/summary_dataset_mean.csv")

tsf = tsf.dropna(axis=1)
num_cols = tsf.select_dtypes(include='number').columns

tsf = tsf.drop(
    columns=num_cols[(tsf[num_cols].abs() > 1e9).any()]
)
constant_features = [col for col in tsf.columns if tsf[col].nunique() <= 1]
tsf = tsf.drop(columns=constant_features)
tsf

# for name, feats in [("tsfresh", tsf), ("catch22", c22), ("summary", sm)]:
for name, feats in [("tsfresh", tsf), ("catch22", c22), ("minirocket", mr), ("summary", sm)]:
    res = feature_rank_alignment(feats, ranks_aggr, feat_metric="cosine", feat_scale=True)
    print(name, res)

tsfresh {'spearman_corr(distance_rank, distance_feat)': 0.12662249930085465, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 1.2266023025295428e-23}
catch22 {'spearman_corr(distance_rank, distance_feat)': 0.04497602271077849, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.00038955005927723775}
minirocket {'spearman_corr(distance_rank, distance_feat)': 0.09163025383843237, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 4.55166226806828e-13}
summary {'spearman_corr(distance_rank, distance_feat)': 0.09383902435838086, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 1.2319995555190174e-13}


In [14]:
tsf = pd.read_csv("data/datasets_features/apriori_meta_tsfresh_probe.csv")
c22 = pd.read_csv("data/datasets_features/apriori_meta_catch22_probe.csv")
sm  = pd.read_csv("data/datasets_features/apriori_meta_summary_probe.csv")

for name, feats in [("tsfresh", tsf), ("catch22", c22), ("summary", sm)]:
    res = feature_rank_alignment(feats, ranks_aggr, feat_metric="cosine", feat_scale=True)
    print(name, res)

tsfresh {'spearman_corr(distance_rank, distance_feat)': 0.02937200675824927, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.02057081520069764}
catch22 {'spearman_corr(distance_rank, distance_feat)': 0.010461296030276313, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.4095747866417505}
summary {'spearman_corr(distance_rank, distance_feat)': 0.019951834531015837, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.11574776066844422}


In [15]:
features = pd.read_csv(Path('data/datasets_features/features.csv'), index_col=0)
features = features.rename(columns={'Name': 'dataset'})
features = features.set_index('dataset').loc[chosen_datasets, :]
features = features.reset_index()
features

,dataset,entropy,gini,number_of_classes,size
0,Adiac,5.202000,0.972675,1.0,781.0
1,ArrowHead,1.576854,0.662833,1.0,211.0
2,Beef,2.321928,0.800000,1.0,60.0
3,BeetleFly,1.000000,0.500000,0.0,40.0
4,BirdChicken,1.000000,0.500000,0.0,40.0
...,...,...,...,...,...
107,EOGHorizontalSignal,3.584885,0.916658,1.0,724.0
108,EOGVerticalSignal,3.584885,0.916658,1.0,724.0
109,Rock,1.925301,0.723673,1.0,70.0
110,Crop,4.584963,0.958333,1.0,24000.0


In [16]:
for name, feats in [("SF", features)]:
    res = feature_rank_alignment(feats, ranks_aggr, feat_metric="cosine", feat_scale=True)
    print(name, res)

SF {'spearman_corr(distance_rank, distance_feat)': -0.01035163987164789, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.41450144350471885}


In [ ]:
def feature_rank_alignment_with_pca(
    feats_ds,
    ranks_aggr,
    feat_metric="cosine",
    feat_scale=True,
    use_pca=True,
    pca_var=0.99,
    random_state=42,
    ):
    
    D_rank, datasets_order = compute_rank_distance_matrix(ranks_aggr)

    f = feats_ds.copy()
    
    f["dataset"] = f["dataset"].astype(str)
    
    f = f.set_index("dataset").reindex(datasets_order)

    X = f.to_numpy(dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if feat_scale:
        X = StandardScaler().fit_transform(X)

    if use_pca:
        pca = PCA(n_components=pca_var, svd_solver="full", random_state=random_state)
        X = pca.fit_transform(X)
        n_comp = X.shape[1]
    else:
        n_comp = X.shape[1]

    D_feat = pairwise_distances(X, metric=feat_metric)

    v_rank = _upper_tri_vals(D_rank)
    v_feat = _upper_tri_vals(D_feat)

    corr = spearmanr(v_rank, v_feat).correlation
    if not np.isfinite(corr):
        corr = 0.0

    return {
        "spearman_corr(distance_rank, distance_feat)": float(corr),
        "n_components": int(n_comp),
        "pca_used": use_pca,
        "pca_var": pca_var if use_pca else None,
    }

In [18]:
tsf = pd.read_csv("data/datasets_features/dataset_level/tsfresh_dataset_mean.csv")
c22 = pd.read_csv("data/datasets_features/dataset_level/catch22_dataset_mean.csv")
mr  = pd.read_csv("data/datasets_features/dataset_level/minirocket_dataset_mean.csv")
sm  = pd.read_csv("data/datasets_features/dataset_level/summary_dataset_mean.csv")
tsf.head()

,dataset,v__variance_larger_than_standard_deviation,v__has_duplicate_max,v__has_duplicate_min,v__has_duplicate,v__sum_values,v__abs_energy,v__mean_abs_change,v__mean_change,v__mean_second_derivative_central,...,v__fourier_entropy__bins_5,v__fourier_entropy__bins_10,v__fourier_entropy__bins_100,v__permutation_entropy__dimension_3__tau_1,v__permutation_entropy__dimension_4__tau_1,v__permutation_entropy__dimension_5__tau_1,v__permutation_entropy__dimension_6__tau_1,v__permutation_entropy__dimension_7__tau_1,v__query_similarity_count__query_None__threshold_0.0,v__mean_n_absolute_max__number_of_maxima_7
0,ACSF1,0.385,0.010000,0.570000,1.000000,1.397573e-08,1460.000000,1.160304,0.000019,-0.000847,...,0.173448,0.208679,0.394857,1.418109,1.658544,2.182301,2.678302,3.107743,NaN,2.825552
1,Adiac,0.000,0.002561,0.000000,0.040973,1.595424e-08,175.000000,0.070771,-0.000006,0.000125,...,0.168004,0.176986,0.223225,0.879698,1.039076,1.177104,1.299305,1.411905,NaN,1.698952
2,ArrowHead,0.000,0.142180,0.018957,0.909953,1.318861e-08,250.000000,0.054496,0.000015,-0.000024,...,0.113604,0.141103,0.245041,1.214883,1.688473,2.082949,2.389370,2.631711,NaN,1.885942
3,BME,0.000,0.211111,0.105556,1.000000,2.238065e+01,56.608056,0.034502,0.000011,0.000012,...,0.224622,0.256778,0.561132,1.668032,2.799389,3.756006,4.259495,4.490649,NaN,1.041510
4,Beef,0.000,0.000000,0.000000,0.916667,-1.127247e-08,469.000000,0.037038,-0.003015,0.000002,...,0.112057,0.184807,0.411851,1.281744,1.912363,2.498251,2.984730,3.366054,NaN,2.572998


In [19]:
tsf = tsf.dropna(axis=1)
num_cols = tsf.select_dtypes(include='number').columns

tsf = tsf.drop(
    columns=num_cols[(tsf[num_cols].abs() > 1e9).any()]
)
constant_features = [col for col in tsf.columns if tsf[col].nunique() <= 1]
tsf = tsf.drop(columns=constant_features)
tsf

,dataset,v__variance_larger_than_standard_deviation,v__has_duplicate_max,v__has_duplicate_min,v__has_duplicate,v__sum_values,v__mean_abs_change,v__mean_change,v__mean_second_derivative_central,v__median,...,v__fourier_entropy__bins_3,v__fourier_entropy__bins_5,v__fourier_entropy__bins_10,v__fourier_entropy__bins_100,v__permutation_entropy__dimension_3__tau_1,v__permutation_entropy__dimension_4__tau_1,v__permutation_entropy__dimension_5__tau_1,v__permutation_entropy__dimension_6__tau_1,v__permutation_entropy__dimension_7__tau_1,v__mean_n_absolute_max__number_of_maxima_7
0,ACSF1,0.385,0.010000,0.570000,1.000000,1.397573e-08,1.160304,0.000019,-0.000847,-0.542405,...,0.077859,0.173448,0.208679,0.394857,1.418109,1.658544,2.182301,2.678302,3.107743,2.825552
1,Adiac,0.000,0.002561,0.000000,0.040973,1.595424e-08,0.070771,-0.000006,0.000125,-0.098502,...,0.073411,0.168004,0.176986,0.223225,0.879698,1.039076,1.177104,1.299305,1.411905,1.698952
2,ArrowHead,0.000,0.142180,0.018957,0.909953,1.318861e-08,0.054496,0.000015,-0.000024,0.078746,...,0.083061,0.113604,0.141103,0.245041,1.214883,1.688473,2.082949,2.389370,2.631711,1.885942
3,BME,0.000,0.211111,0.105556,1.000000,2.238065e+01,0.034502,0.000011,0.000012,0.093862,...,0.175942,0.224622,0.256778,0.561132,1.668032,2.799389,3.756006,4.259495,4.490649,1.041510
4,Beef,0.000,0.000000,0.000000,0.916667,-1.127247e-08,0.037038,-0.003015,0.000002,0.081233,...,0.065581,0.112057,0.184807,0.411851,1.281744,1.912363,2.498251,2.984730,3.366054,2.572998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,Wine,0.000,0.000000,0.000000,1.000000,2.101288e-08,0.073393,-0.015458,0.000256,-0.337871,...,0.097664,0.101614,0.246662,0.416471,1.007531,1.349472,1.694412,2.046765,2.383959,3.007583
108,WordSynonyms,0.000,0.000000,0.000000,0.057459,1.124446e-08,0.066198,0.000270,-0.000196,-0.325636,...,0.136939,0.187805,0.260067,0.490834,0.905597,1.149245,1.393245,1.635372,1.872081,2.938938
109,Worms,0.000,0.100775,0.093023,0.992248,4.282148e-08,0.057811,0.000116,-0.000003,-0.006140,...,0.099287,0.138546,0.183107,0.351580,1.506146,2.415932,3.346672,4.137341,4.669882,2.474004
110,WormsTwoClass,0.000,0.100775,0.093023,0.992248,4.282148e-08,0.057811,0.000116,-0.000003,-0.006140,...,0.099287,0.138546,0.183107,0.351580,1.506146,2.415932,3.346672,4.137341,4.669882,2.474004


In [20]:
for var in [0.90, 0.95, 0.99]:
    res = feature_rank_alignment_with_pca(tsf, ranks_aggr, use_pca=True, pca_var=var)
    print(f"PCA {var}: corr={res['spearman_corr(distance_rank, distance_feat)']:.3f}, "
          f"n_comp={res['n_components']}")

PCA 0.9: corr=0.127, n_comp=16
PCA 0.95: corr=0.125, n_comp=25
PCA 0.99: corr=0.127, n_comp=46


In [21]:
for metric in ["euclidean", "cosine", "correlation"]:
    res = feature_rank_alignment(c22, ranks_aggr, feat_metric=metric, feat_scale=True)
    print(metric, res)

euclidean {'spearman_corr(distance_rank, distance_feat)': 0.023467449312508295, 'n_datasets': 112, 'feat_metric': 'euclidean', 'feat_scaled': True, 'p_value': 0.06429996932171994}
cosine {'spearman_corr(distance_rank, distance_feat)': 0.04497602271077849, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.00038955005927723775}
correlation {'spearman_corr(distance_rank, distance_feat)': 0.05056252710594704, 'n_datasets': 112, 'feat_metric': 'correlation', 'feat_scaled': True, 'p_value': 6.658201290960029e-05}


In [22]:
tsf = pd.read_csv("data/datasets_features/dataset_level/tsfresh_dataset_mean.csv")

In [23]:
tsf = tsf.dropna(axis=1)
num_cols = tsf.select_dtypes(include='number').columns

tsf = tsf.drop(
    columns=num_cols[(tsf[num_cols].abs() > 1e9).any()]
)
constant_features = [col for col in tsf.columns if tsf[col].nunique() <= 1]
tsf = tsf.drop(columns=constant_features)
tsf

,dataset,v__variance_larger_than_standard_deviation,v__has_duplicate_max,v__has_duplicate_min,v__has_duplicate,v__sum_values,v__mean_abs_change,v__mean_change,v__mean_second_derivative_central,v__median,...,v__fourier_entropy__bins_3,v__fourier_entropy__bins_5,v__fourier_entropy__bins_10,v__fourier_entropy__bins_100,v__permutation_entropy__dimension_3__tau_1,v__permutation_entropy__dimension_4__tau_1,v__permutation_entropy__dimension_5__tau_1,v__permutation_entropy__dimension_6__tau_1,v__permutation_entropy__dimension_7__tau_1,v__mean_n_absolute_max__number_of_maxima_7
0,ACSF1,0.385,0.010000,0.570000,1.000000,1.397573e-08,1.160304,0.000019,-0.000847,-0.542405,...,0.077859,0.173448,0.208679,0.394857,1.418109,1.658544,2.182301,2.678302,3.107743,2.825552
1,Adiac,0.000,0.002561,0.000000,0.040973,1.595424e-08,0.070771,-0.000006,0.000125,-0.098502,...,0.073411,0.168004,0.176986,0.223225,0.879698,1.039076,1.177104,1.299305,1.411905,1.698952
2,ArrowHead,0.000,0.142180,0.018957,0.909953,1.318861e-08,0.054496,0.000015,-0.000024,0.078746,...,0.083061,0.113604,0.141103,0.245041,1.214883,1.688473,2.082949,2.389370,2.631711,1.885942
3,BME,0.000,0.211111,0.105556,1.000000,2.238065e+01,0.034502,0.000011,0.000012,0.093862,...,0.175942,0.224622,0.256778,0.561132,1.668032,2.799389,3.756006,4.259495,4.490649,1.041510
4,Beef,0.000,0.000000,0.000000,0.916667,-1.127247e-08,0.037038,-0.003015,0.000002,0.081233,...,0.065581,0.112057,0.184807,0.411851,1.281744,1.912363,2.498251,2.984730,3.366054,2.572998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,Wine,0.000,0.000000,0.000000,1.000000,2.101288e-08,0.073393,-0.015458,0.000256,-0.337871,...,0.097664,0.101614,0.246662,0.416471,1.007531,1.349472,1.694412,2.046765,2.383959,3.007583
108,WordSynonyms,0.000,0.000000,0.000000,0.057459,1.124446e-08,0.066198,0.000270,-0.000196,-0.325636,...,0.136939,0.187805,0.260067,0.490834,0.905597,1.149245,1.393245,1.635372,1.872081,2.938938
109,Worms,0.000,0.100775,0.093023,0.992248,4.282148e-08,0.057811,0.000116,-0.000003,-0.006140,...,0.099287,0.138546,0.183107,0.351580,1.506146,2.415932,3.346672,4.137341,4.669882,2.474004
110,WormsTwoClass,0.000,0.100775,0.093023,0.992248,4.282148e-08,0.057811,0.000116,-0.000003,-0.006140,...,0.099287,0.138546,0.183107,0.351580,1.506146,2.415932,3.346672,4.137341,4.669882,2.474004


In [24]:
ranks_aggr

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,dataset
Adiac,2.000000,16.066667,11.700000,15.733333,1.000000,6.666667,29.066667,3.933333,27.966667,22.133333,...,3.600000,6.633333,13.466667,18.333333,8.066667,20.933333,31.133333,22.833333,11.733333,Adiac
ArrowHead,2.666667,21.166667,20.200000,11.800000,5.600000,4.800000,13.066667,19.200000,7.900000,25.433333,...,18.433333,5.800000,29.900000,26.300000,7.000000,2.433333,27.333333,24.433333,25.733333,ArrowHead
Beef,3.200000,22.766667,11.600000,27.466667,15.766667,3.166667,28.066667,4.466667,28.166667,20.133333,...,4.766667,11.933333,14.166667,10.633333,14.900000,20.700000,25.400000,19.800000,7.966667,Beef
BeetleFly,5.166667,17.500000,24.566667,14.900000,8.566667,14.066667,18.500000,6.233333,18.366667,28.766667,...,4.300000,6.533333,23.333333,27.000000,9.900000,1.000000,25.533333,22.366667,29.600000,BeetleFly
BirdChicken,9.633333,16.633333,27.800000,12.066667,4.300000,14.700000,25.400000,12.300000,21.866667,27.633333,...,3.366667,3.800000,24.966667,23.366667,8.733333,4.866667,18.466667,17.800000,33.166667,BirdChicken
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
EOGHorizontalSignal,2.033333,24.166667,6.500000,20.200000,1.000000,11.566667,23.566667,20.800000,17.666667,23.100000,...,3.733333,8.333333,10.900000,31.666667,7.933333,14.800000,14.500000,7.133333,9.866667,EOGHorizontalSignal
EOGVerticalSignal,2.766667,26.233333,7.433333,24.266667,1.000000,11.800000,27.600000,19.600000,14.066667,23.566667,...,2.900000,6.466667,13.666667,32.533333,10.533333,12.166667,13.533333,3.533333,8.333333,EOGVerticalSignal
Rock,4.733333,16.533333,16.400000,20.266667,9.633333,12.266667,23.733333,12.200000,22.900000,25.266667,...,6.733333,10.533333,26.200000,20.500000,12.066667,6.900000,27.300000,23.000000,18.333333,Rock
Crop,6.700000,13.633333,2.100000,32.233333,5.833333,2.266667,33.466667,11.400000,27.766667,27.666667,...,7.866667,17.100000,8.966667,23.300000,15.766667,19.100000,21.033333,12.766667,4.266667,Crop


In [26]:
for metric in ["euclidean", "cosine", "correlation"]:
    res = feature_rank_alignment(tsf, ranks_aggr, feat_metric=metric, feat_scale=True)
    print(metric, res)

euclidean {'spearman_corr(distance_rank, distance_feat)': 0.15369051919472518, 'n_datasets': 112, 'feat_metric': 'euclidean', 'feat_scaled': True, 'p_value': 3.635134192257606e-34}
cosine {'spearman_corr(distance_rank, distance_feat)': 0.12662249930085465, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 1.2266023025295428e-23}
correlation {'spearman_corr(distance_rank, distance_feat)': 0.11476665207645649, 'n_datasets': 112, 'feat_metric': 'correlation', 'feat_scaled': True, 'p_value': 1.1204813787683317e-19}


In [27]:
for metric in ["euclidean", "cosine", "correlation"]:
    res = feature_rank_alignment(mr, ranks_aggr, feat_metric=metric, feat_scale=True)
    print(metric, res)

euclidean {'spearman_corr(distance_rank, distance_feat)': 0.16278856967885447, 'n_datasets': 112, 'feat_metric': 'euclidean', 'feat_scaled': True, 'p_value': 3.554135587569013e-38}
cosine {'spearman_corr(distance_rank, distance_feat)': 0.09163025383843237, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 4.55166226806828e-13}
correlation {'spearman_corr(distance_rank, distance_feat)': 0.09179686559630035, 'n_datasets': 112, 'feat_metric': 'correlation', 'feat_scaled': True, 'p_value': 4.1287747294442277e-13}


In [28]:
for metric in ["euclidean", "cosine", "correlation"]:
    res = feature_rank_alignment(sm, ranks_aggr, feat_metric=metric, feat_scale=True)
    print(metric, res)

euclidean {'spearman_corr(distance_rank, distance_feat)': 0.12173053438180147, 'n_datasets': 112, 'feat_metric': 'euclidean', 'feat_scaled': True, 'p_value': 5.8956982368149475e-22}
cosine {'spearman_corr(distance_rank, distance_feat)': 0.09383902435838086, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 1.2319995555190174e-13}
correlation {'spearman_corr(distance_rank, distance_feat)': 0.08557671368500444, 'n_datasets': 112, 'feat_metric': 'correlation', 'feat_scaled': True, 'p_value': 1.3986233536632172e-11}


In [29]:
for metric in ["euclidean", "cosine", "correlation"]:
    res = feature_rank_alignment(c22, ranks_aggr, feat_metric=metric, feat_scale=True)
    print(metric, res)

euclidean {'spearman_corr(distance_rank, distance_feat)': 0.023467449312508295, 'n_datasets': 112, 'feat_metric': 'euclidean', 'feat_scaled': True, 'p_value': 0.06429996932171994}
cosine {'spearman_corr(distance_rank, distance_feat)': 0.04497602271077849, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.00038955005927723775}
correlation {'spearman_corr(distance_rank, distance_feat)': 0.05056252710594704, 'n_datasets': 112, 'feat_metric': 'correlation', 'feat_scaled': True, 'p_value': 6.658201290960029e-05}


In [30]:
# euclidean {'spearman_corr(distance_rank, distance_feat)': 0.029611318774038612, 'n_datasets': 109, 'feat_metric': 'euclidean', 'feat_scaled': True}
# cosine {'spearman_corr(distance_rank, distance_feat)': 0.050173766997252366, 'n_datasets': 109, 'feat_metric': 'cosine', 'feat_scaled': True}
# correlation {'spearman_corr(distance_rank, distance_feat)': 0.05477536709485996, 

In [31]:
lm = pd.read_csv("data/datasets_features/landmarking_raw.csv")

In [32]:
for metric in ["euclidean", "cosine", "correlation"]:
    res = feature_rank_alignment(lm, ranks_aggr, feat_metric=metric, feat_scale=True)
    print(metric, res)

euclidean {'spearman_corr(distance_rank, distance_feat)': -0.008986146677372598, 'n_datasets': 112, 'feat_metric': 'euclidean', 'feat_scaled': True, 'p_value': 0.4787253775973689}
cosine {'spearman_corr(distance_rank, distance_feat)': 0.05363203664903143, 'n_datasets': 112, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 2.3303694562716407e-05}
correlation {'spearman_corr(distance_rank, distance_feat)': 0.045462250706349516, 'n_datasets': 112, 'feat_metric': 'correlation', 'feat_scaled': True, 'p_value': 0.0003364973401543087}


In [33]:
data_rec = pd.read_csv(Path('data/rec_sys_datasets/Datafeatures_and_metrics.csv')).sort_values(by='Dataset')

datasets_recsys = data_rec['Dataset'].unique()
datasets_recsys, datasets_recsys.shape

temp = data_rec.groupby(["Method", "Dataset"])["Value"].mean().unstack("Dataset")
# temp = metrics_data.groupby(["Method", "Dataset"])["Value"].mean().unstack("Dataset")
ranks_aggr_recsys = temp.rank(method="min", ascending=False)
ranks_aggr_recsys.index.name = None
ranks_aggr_recsys.columns.name = None
# ranks_aggr_recsys = ranks_aggr_recsys.reset_index(drop=True)
ranks_aggr_recsys = ranks_aggr_recsys
ranks_aggr_recsys

ranks_recsys = {}
for ds in datasets_recsys:
    df_ds = pd.DataFrame({
        "model": ranks_aggr_recsys.index,
    })
    for i in range(30):
        df_ds[str(i)] = ranks_aggr_recsys[ds].values
    ranks_recsys[ds] = df_ds
ranks_recsys


recsys_features = (data_rec.drop(columns=['Method', 'Value']).drop_duplicates(subset=['Dataset'])
                   .sort_values(by='Dataset').reset_index(drop=True))
recsys_features = recsys_features.rename(columns={"Dataset": "dataset"})


In [34]:
ranks_aggr_recsys

,amazon_cds,amazon_finefoods,amazon_mi,amazon_tv,beeradvocate,brightkite,dianping,douban_books,douban_movies,douban_music,...,mts_library,netflix,ratebeer,reddit,rekko,retail,sber_smm,sber_zvuk,tafeng,yelp
ALS,5.0,1.0,5.0,5.0,1.0,8.0,7.0,6.0,6.0,6.0,...,5.0,5.0,4.0,7.0,2.0,7.0,4.0,3.0,7.0,3.0
BPR,8.0,9.0,7.0,8.0,9.0,9.0,9.0,1.0,2.0,4.0,...,9.0,8.0,8.0,9.0,9.0,8.0,9.0,9.0,1.0,8.0
EASE,2.0,2.0,4.0,2.0,2.0,1.0,2.0,4.0,4.0,9.0,...,1.0,1.0,3.0,2.0,1.0,1.0,2.0,1.0,6.0,1.0
ItemKNN,6.0,8.0,2.0,6.0,4.0,7.0,5.0,8.0,7.0,7.0,...,4.0,9.0,5.0,6.0,7.0,5.0,3.0,5.0,5.0,7.0
LightFM,7.0,7.0,8.0,7.0,6.0,6.0,6.0,5.0,8.0,8.0,...,3.0,3.0,6.0,8.0,3.0,9.0,6.0,4.0,2.0,6.0
LightGCL,1.0,5.0,1.0,4.0,8.0,5.0,8.0,9.0,9.0,3.0,...,8.0,6.0,9.0,4.0,8.0,6.0,7.0,8.0,4.0,5.0
LightGCN,4.0,3.0,3.0,1.0,7.0,3.0,4.0,7.0,1.0,1.0,...,6.0,4.0,2.0,3.0,6.0,3.0,8.0,6.0,9.0,4.0
MultiVAE,3.0,6.0,6.0,3.0,5.0,4.0,3.0,2.0,5.0,5.0,...,7.0,2.0,7.0,1.0,5.0,4.0,5.0,2.0,3.0,2.0
SLIM,9.0,4.0,9.0,9.0,3.0,2.0,1.0,3.0,3.0,2.0,...,2.0,7.0,1.0,5.0,4.0,2.0,1.0,7.0,8.0,9.0


In [35]:
ranks_aggr_r = pd.DataFrame(columns=datasets_recsys)

for d in datasets_recsys:
    ranks_aggr_r[d] = ranks_recsys[d].drop(columns=['model']).mean(axis=1)


ranks_aggr_r = ranks_aggr_r.transpose()
ranks_aggr_r = ranks_aggr_r.reset_index(names='dataset')
ranks_aggr_r

,dataset,0,1,2,3,4,5,6,7,8
0,amazon_cds,5.0,8.0,2.0,6.0,7.0,1.0,4.0,3.0,9.0
1,amazon_finefoods,1.0,9.0,2.0,8.0,7.0,5.0,3.0,6.0,4.0
2,amazon_mi,5.0,7.0,4.0,2.0,8.0,1.0,3.0,6.0,9.0
3,amazon_tv,5.0,8.0,2.0,6.0,7.0,4.0,1.0,3.0,9.0
4,beeradvocate,1.0,9.0,2.0,4.0,6.0,8.0,7.0,5.0,3.0
5,brightkite,8.0,9.0,1.0,7.0,6.0,5.0,3.0,4.0,2.0
6,dianping,7.0,9.0,2.0,5.0,6.0,8.0,4.0,3.0,1.0
7,douban_books,6.0,1.0,4.0,8.0,5.0,9.0,7.0,2.0,3.0
8,douban_movies,6.0,2.0,4.0,7.0,8.0,9.0,1.0,5.0,3.0
9,douban_music,6.0,4.0,9.0,7.0,8.0,3.0,1.0,5.0,2.0


In [36]:
for metric in ["euclidean", "cosine", "correlation"]:
    res = feature_rank_alignment(recsys_features, ranks_aggr_r, feat_metric=metric, feat_scale=True)
    print(metric, res)

euclidean {'spearman_corr(distance_rank, distance_feat)': 0.06970881583818656, 'n_datasets': 30, 'feat_metric': 'euclidean', 'feat_scaled': True, 'p_value': 0.14664753323790183}
cosine {'spearman_corr(distance_rank, distance_feat)': 0.08351113498417687, 'n_datasets': 30, 'feat_metric': 'cosine', 'feat_scaled': True, 'p_value': 0.08189567582342039}
correlation {'spearman_corr(distance_rank, distance_feat)': 0.08830346617145791, 'n_datasets': 30, 'feat_metric': 'correlation', 'feat_scaled': True, 'p_value': 0.06576658304426308}


In [ ]:
def compare_representations_global_geometry(
    representations,
    metric="cosine",
    scale=True,
    corr="spearman",
    dataset_col="dataset",
):

    rep_dfs = {}
    dataset_sets = []

    for name, X in representations.items():
        if isinstance(X, pd.DataFrame):
            df = X.copy()
            if dataset_col in df.columns:
                df[dataset_col] = df[dataset_col].astype(str)
                df = df.set_index(dataset_col)
            else:
                df.index = df.index.astype(str)
            rep_dfs[name] = df
            dataset_sets.append(set(df.index))
        else:
            raise ValueError("Pass DataFrames (with dataset column or index) to avoid silent misalignment.")

    common = set.intersection(*dataset_sets)
    if len(common) == 0:
        raise ValueError("No common datasets across representations.")

    datasets_order = np.array(sorted(common), dtype=str)

    names = list(rep_dfs.keys())
    dist_vecs = {}

    for name in names:
        df = rep_dfs[name].loc[datasets_order]

        X = df.to_numpy(dtype=float)

        if name == "ranks":
            
            n = X.shape[0]
            D = np.zeros((n, n), dtype=float)
            
            for i in range(n):
                for j in range(i + 1, n):
                    
                    rho = spearmanr(X[i], X[j]).correlation
                    if not np.isfinite(rho):
                        rho = 0.0
                        
                    D[i, j] = D[j, i] = 1.0 - float(rho)
        else:
            
            if scale:
                X = StandardScaler().fit_transform(X)
                
            D = pairwise_distances(X, metric=metric)

        dist_vecs[name] = D[np.triu_indices_from(D, k=1)]

    R = np.zeros((len(names), len(names)), dtype=float)

    for i, ni in enumerate(names):
        for j, nj in enumerate(names):
            if corr == "spearman":
                R[i, j] = spearmanr(dist_vecs[ni], dist_vecs[nj]).correlation
            elif corr == "pearson":
                R[i, j] = np.corrcoef(dist_vecs[ni], dist_vecs[nj])[0, 1]

    return pd.DataFrame(R, index=names, columns=names), datasets_order

In [38]:
representations = {
    "catch22": c22,
    "tsfresh": tsf,
    "summary": sm,
    "minirocket": mr,
    "landmarking": lm,
    "ranks": ranks_aggr
}

R = compare_representations_global_geometry(
    representations,
    metric="cosine",
    scale=True,
    corr="spearman",
)

print("cosine")
print(R[0])

cosine
              catch22   tsfresh   summary  minirocket  landmarking     ranks
catch22      1.000000  0.645749  0.566967    0.276211     0.094389  0.044976
tsfresh      0.645749  1.000000  0.657711    0.337487     0.142565  0.126624
summary      0.566967  0.657711  1.000000    0.266394     0.118461  0.093839
minirocket   0.276211  0.337487  0.266394    1.000000     0.205727  0.091630
landmarking  0.094389  0.142565  0.118461    0.205727     1.000000  0.053633
ranks        0.044976  0.126624  0.093839    0.091630     0.053633  1.000000


In [39]:
representations = {
    "catch22": c22,
    "tsfresh": tsf,
    "summary": sm,
    "minirocket": mr,
    "landmarking": lm,
    "ranks": ranks_aggr
}

R = compare_representations_global_geometry(
    representations,
    metric="euclidean",
    scale=True,
    corr="spearman",
)

print("euclidean")
print(R[0])

euclidean
              catch22   tsfresh   summary  minirocket  landmarking     ranks
catch22      1.000000  0.471167  0.539792    0.375339     0.144638  0.023468
tsfresh      0.471167  1.000000  0.706124    0.464160     0.101129  0.153691
summary      0.539792  0.706124  1.000000    0.483636     0.120081  0.121731
minirocket   0.375339  0.464160  0.483636    1.000000     0.199506  0.162789
landmarking  0.144638  0.101129  0.120081    0.199506     1.000000 -0.008986
ranks        0.023468  0.153691  0.121731    0.162789    -0.008986  1.000000


In [ ]:
def zscore_columns(X, eps=1e-9):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0, keepdims=True)
    std = X.std(axis=0, keepdims=True)
    return (X - mean) / (std + eps)

def as_2d_float(a):
    a = np.asarray(a, dtype=float)
    if a.ndim == 1:
        a = a[:, None]
    return a

def center_features(X):
    X = as_2d_float(X)
    return X - X.mean(axis=0, keepdims=True)

def gram_linear(X):
    X = as_2d_float(X)
    return X @ X.T

def center_gram(K):
    K = np.asarray(K, dtype=float)
    n = K.shape[0]
    one = np.ones((n, n), dtype=float) / n
    return K - one @ K - K @ one + one @ K @ one

def _hsic(K, L):
    Kc = center_gram(K)
    Lc = center_gram(L)
    return float(np.sum(Kc * Lc))

def linear_cka(X, Y):
    Xc = center_features(X)
    Yc = center_features(Y)
    K = gram_linear(Xc)
    L = gram_linear(Yc)

    hsic_xy = _hsic(K, L)
    hsic_xx = _hsic(K, K)
    hsic_yy = _hsic(L, L)
    denom = np.sqrt(hsic_xx * hsic_yy)
    return float(hsic_xy / denom) if denom > 0 else np.nan

def rsm_spearman(X, Y):
    Xc = center_features(X)
    Yc = center_features(Y)
    K = center_gram(gram_linear(Xc))
    L = center_gram(gram_linear(Yc))
    iu = np.triu_indices(K.shape[0], k=1)
    return float(spearmanr(K[iu], L[iu]).correlation)

def align_rank_and_features(
    ranks_aggr,
    feats_df,
    dataset_col="dataset",
):

    if dataset_col not in ranks_aggr.columns:
        raise ValueError(f"ranks_aggr must contain column '{dataset_col}'")
    if dataset_col not in feats_df.columns:
        raise ValueError(f"feats_df must contain column '{dataset_col}'")

    r = ranks_aggr.drop_duplicates(subset=[dataset_col], keep="first").copy()
    f = feats_df.drop_duplicates(subset=[dataset_col], keep="first").copy()

    common = sorted(set(r[dataset_col].values).intersection(set(f[dataset_col].values)))
    if len(common) == 0:
        raise ValueError("No common datasets between ranks_aggr and feats_df")

    r_num = r[r[dataset_col].isin(common)].set_index(dataset_col)
    f_num = f[f[dataset_col].isin(common)].set_index(dataset_col)

    r_num = r_num.select_dtypes(include=[np.number]).loc[common]
    f_num = f_num.select_dtypes(include=[np.number]).loc[common]

    X_rank = r_num.to_numpy(dtype=float)
    X_feat = f_num.to_numpy(dtype=float)

    return X_rank, X_feat, common

def cka_rsm_rank_vs_features(ranks_aggr, feats_df, dataset_col="dataset"):
    
    X_rank, X_feat, common = align_rank_and_features(ranks_aggr, feats_df, dataset_col=dataset_col)
    
    X_rank = zscore_columns(X_rank)
    
    X_feat = zscore_columns(X_feat)
    
    return {
        "n_common": len(common),
        "cka_linear": linear_cka(X_rank, X_feat),
        "rsm_spearman": rsm_spearman(X_rank, X_feat),
        "common_datasets": common,
    }


In [41]:
result = cka_rsm_rank_vs_features(ranks_aggr, tsf, dataset_col="dataset")
print(result["n_common"], result["cka_linear"], result["rsm_spearman"])

112 0.11015132873919863 0.06791577606112648


In [42]:
result = cka_rsm_rank_vs_features(ranks_aggr, c22, dataset_col="dataset")
print(result["n_common"], result["cka_linear"], result["rsm_spearman"])

112 0.14107373714316354 0.05588798361111536


In [43]:
result = cka_rsm_rank_vs_features(ranks_aggr, mr, dataset_col="dataset")
print(result["n_common"], result["cka_linear"], result["rsm_spearman"])

112 0.13400576789073593 0.06973239609386989


In [44]:
result = cka_rsm_rank_vs_features(ranks_aggr, lm, dataset_col="dataset")
print(result["n_common"], result["cka_linear"], result["rsm_spearman"])

112 0.2103954168191669 0.15314045258303152


In [45]:
result = cka_rsm_rank_vs_features(ranks_aggr, sm, dataset_col="dataset")
print(result["n_common"], result["cka_linear"], result["rsm_spearman"])

112 0.09844627389479883 0.0387539885426009


In [ ]:
def pairwise_cka_rsm(
    representations,
    dataset_col="dataset",
    metric="cka",
):
    names = list(representations.keys())
    n = len(names)

    M = pd.DataFrame(
        np.zeros((n, n)),
        index=names,
        columns=names,
        dtype=float,
    )

    for i, ni in enumerate(names):
        for j, nj in enumerate(names):
            if j < i:
                M.loc[ni, nj] = M.loc[nj, ni]
                continue

            res = cka_rsm_rank_vs_features(
                representations[ni],
                representations[nj],
                dataset_col=dataset_col,
            )

            if metric == "cka":
                val = res["cka_linear"]
            elif metric == "rsm":
                val = res["rsm_spearman"]

            M.loc[ni, nj] = val
            
            M.loc[nj, ni] = val

    print(f"{metric.upper()} similarity")
    print(M.round(4))
    return M

In [47]:
representations = {
    "catch22": c22,
    "tsfresh": tsf,
    "summary": sm,
    "minirocket": mr,
    "landmarking": lm,
    "ranks": ranks_aggr,
}


df = pairwise_cka_rsm(
    dataset_col = "dataset",
    metric = "cka",
    representations=representations,
)

CKA similarity
             catch22  tsfresh  summary  minirocket  landmarking   ranks
catch22       1.0000   0.2255   0.4394      0.2057       0.1261  0.1411
tsfresh       0.2255   1.0000   0.6137      0.2551       0.0964  0.1102
summary       0.4394   0.6137   1.0000      0.2257       0.0923  0.0984
minirocket    0.2057   0.2551   0.2257      1.0000       0.2164  0.1340
landmarking   0.1261   0.0964   0.0923      0.2164       1.0000  0.2104
ranks         0.1411   0.1102   0.0984      0.1340       0.2104  1.0000
